In [ ]:
import pandas as pd
import ast
import numpy as np
import torch

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer
)

DATA_PATH = "../final_training_data/final_resume_skill_ner_dataset_fixed.csv"
MODEL_PATH = "../../../ml_models/skill_ner_model"

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.13.0+cu126
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 A Laptop GPU


In [12]:
df = pd.read_csv(DATA_PATH)

df["tokens"] = df["tokens"].apply(ast.literal_eval)
df["ner_tags"] = df["ner_tags"].apply(ast.literal_eval)
df["ner_ids"] = df["ner_ids"].apply(ast.literal_eval)
df["skills"] = df["skills"].apply(ast.literal_eval)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

Dataset shape: (9998, 7)
Columns: ['resume_id', 'resume_text', 'tokens', 'ner_tags', 'skills', 'num_skills', 'ner_ids']


In [13]:
print("Total resumes:", len(df))

print("\nLabel distribution:")
all_labels = [x for row in df["ner_ids"] for x in row]

unique, counts = np.unique(all_labels, return_counts=True)

for label, count in zip(unique, counts):
    print(f"{label}: {count}")

print("\nChecking token/label lengths...")

bad_rows = []

for idx, row in df.iterrows():
    if len(row["tokens"]) != len(row["ner_ids"]):
        bad_rows.append(idx)

print("Invalid rows:", len(bad_rows))

assert len(bad_rows) == 0, "Token/label length mismatch found!"

print("\nDataset integrity: PASSED")

Total resumes: 9998

Label distribution:
0: 243411
1: 50002
2: 20388

Checking token/label lengths...
Invalid rows: 0

Dataset integrity: PASSED


In [14]:
MODEL_NAME = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

print("Tokenizer:", MODEL_NAME)
print("Vocabulary size:", tokenizer.vocab_size)

Tokenizer: bert-base-uncased
Vocabulary size: 30522


In [17]:
def tokenize_and_align_labels(examples):

    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        max_length=256,
        is_split_into_words=True
    )

    all_labels = []

    for i, labels in enumerate(examples["ner_ids"]):

        word_ids = tokenized_inputs.word_ids(
            batch_index=i
        )

        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:

            if word_idx is None:

                label_ids.append(-100)

            elif word_idx != previous_word_idx:

                label_ids.append(labels[word_idx])

            else:

                # If B-SKILL is split into multiple
                # subword tokens, subsequent pieces
                # become I-SKILL.
                if labels[word_idx] == 1:
                    label_ids.append(2)
                else:
                    label_ids.append(labels[word_idx])

            previous_word_idx = word_idx

        all_labels.append(label_ids)

    tokenized_inputs["labels"] = all_labels

    return tokenized_inputs

In [20]:
import os

print("Current folder:")
print(os.getcwd())

print("\nFiles and folders:")
for item in os.listdir():
    print(item)

Current folder:
c:\Users\pushk\full_stack_projects\EDI-final\ml_training\resume_extraction\raw_data_processing

Files and folders:
01_skill_extraction_pipeline.ipynb
final_training.ipynb
skill_ner_model
training.ipynb


In [7]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch
import os

MODEL_PATH = "./skill_ner_model/checkpoint-3000"

print("Files in checkpoint:")
for f in os.listdir(MODEL_PATH):
    print(" -", f)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    use_fast=False
)

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_PATH
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)
model.eval()

print("\n✅ MODEL LOADED")
print("Device:", device)
print("Labels:", model.config.id2label)

Files in checkpoint:
 - config.json
 - model.safetensors
 - optimizer.pt
 - rng_state.pth
 - scaler.pt
 - scheduler.pt
 - tokenizer.json
 - tokenizer_config.json
 - trainer_state.json
 - training_args.bin


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]


✅ MODEL LOADED
Device: cuda
Labels: {0: 'O', 1: 'B-SKILL', 2: 'I-SKILL'}


In [8]:
print(model.config._name_or_path)
print(model.config.id2label)

./skill_ner_model/checkpoint-3000
{0: 'O', 1: 'B-SKILL', 2: 'I-SKILL'}


In [11]:
def predict(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512,
        return_offsets_mapping=True
    )

    offsets = inputs.pop("offset_mapping")[0]

    model_inputs = {
        k: v.to(device)
        for k, v in inputs.items()
    }

    with torch.no_grad():
        outputs = model(**model_inputs)

    predictions = torch.argmax(
        outputs.logits,
        dim=-1
    )[0]

    id2label = model.config.id2label

    entities = []

    current_start = None
    current_end = None
    current_label = None

    for i, pred_id in enumerate(predictions):

        label = id2label[int(pred_id)]

        start, end = offsets[i].tolist()

        if start == end:
            continue

        # ------------------------------------------------
        # B-SKILL = Start of a new skill
        # ------------------------------------------------
        if label.startswith("B-"):

            # Save previous entity
            if current_start is not None:
                entities.append({
                    "skill": text[current_start:current_end].strip(),
                    "label": current_label
                })

            current_start = start
            current_end = end
            current_label = label[2:]

        # ------------------------------------------------
        # I-SKILL = Continue current skill
        # ------------------------------------------------
        elif label.startswith("I-"):

            if current_start is not None:
                current_end = end

        # ------------------------------------------------
        # O = Outside entity
        # ------------------------------------------------
        else:

            if current_start is not None:
                entities.append({
                    "skill": text[current_start:current_end].strip(),
                    "label": current_label
                })

                current_start = None
                current_end = None
                current_label = None

    # Save final entity
    if current_start is not None:
        entities.append({
            "skill": text[current_start:current_end].strip(),
            "label": current_label
        })

    # Remove duplicates
    final_skills = []
    seen = set()

    for entity in entities:

        skill = entity["skill"].strip()

        if skill and skill.lower() not in seen:

            seen.add(skill.lower())

            final_skills.append({
                "skill": skill,
                "label": entity["label"]
            })

    return final_skills

In [12]:
text = """
I am a Full Stack Developer skilled in Python, JavaScript,
React.js, Node.js, Express.js, MongoDB, SQL, Docker,
Git, AWS and Machine Learning.
"""

result = predict(text)

print("INPUT:")
print(text)

print("\n" + "=" * 60)
print("EXTRACTED SKILLS")
print("=" * 60)

if result:
    for item in result:
        print(f"✓ {item['skill']}  →  {item['label']}")
else:
    print("No skills detected.")

INPUT:

I am a Full Stack Developer skilled in Python, JavaScript,
React.js, Node.js, Express.js, MongoDB, SQL, Docker,
Git, AWS and Machine Learning.


EXTRACTED SKILLS
✓ Python  →  SKILL
✓ JavaScript  →  SKILL
✓ MongoDB  →  SKILL
✓ SQL  →  SKILL
✓ Docker  →  SKILL
✓ Git  →  SKILL
✓ AWS  →  SKILL
✓ Machine Learning  →  SKILL
